In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

data = pd.read_csv("magic04.data")
data.columns = [
    'fLength','fWidth','fSize','fConc','fConc1',
    'fAsym','fM3Long','fM3Trans','fAlpha','fDist','class']

print(data.head())
print(data.tail(10))
print(data['class'].value_counts())
# Load data and check first and last rows
# Count how many samples in class (g, h)

    fLength    fWidth   fSize   fConc  fConc1     fAsym  fM3Long  fM3Trans  \
0   31.6036   11.7235  2.5185  0.5303  0.3773   26.2722  23.8238   -9.9574   
1  162.0520  136.0310  4.0612  0.0374  0.0187  116.7410 -64.8580  -45.2160   
2   23.8172    9.5728  2.3385  0.6147  0.3922   27.2107  -6.4633   -7.1513   
3   75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277  28.5525   21.8393   
4   51.6240   21.1502  2.9085  0.2420  0.1340   50.8761  43.1887    9.8145   

    fAlpha    fDist class  
0   6.3609  205.261     g  
1  76.9600  256.788     g  
2  10.4490  116.737     g  
3   4.6480  356.462     g  
4   3.6130  238.098     g  
        fLength   fWidth   fSize   fConc  fConc1     fAsym   fM3Long  \
19009   32.4902  10.6723  2.4742  0.4664  0.2735  -27.0097  -21.1687   
19010   79.5528  44.9929  3.5488  0.1656  0.0900  -39.6213   53.7866   
19011   31.8373  13.8734  2.8251  0.4169  0.1988  -16.4919  -27.1448   
19012  182.5003  76.5568  3.6872  0.1123  0.0666  192.2675   93.0302   
190

In [2]:
data['class'] = data['class'].map({'g':1 , 'h':0})
print(data.head(10))
print(data.tail(10))
# Change class names to numbers: g to 1, h to 0
# Needed for KNN to work

    fLength    fWidth   fSize   fConc  fConc1     fAsym  fM3Long  fM3Trans  \
0   31.6036   11.7235  2.5185  0.5303  0.3773   26.2722  23.8238   -9.9574   
1  162.0520  136.0310  4.0612  0.0374  0.0187  116.7410 -64.8580  -45.2160   
2   23.8172    9.5728  2.3385  0.6147  0.3922   27.2107  -6.4633   -7.1513   
3   75.1362   30.9205  3.1611  0.3168  0.1832   -5.5277  28.5525   21.8393   
4   51.6240   21.1502  2.9085  0.2420  0.1340   50.8761  43.1887    9.8145   
5   48.2468   17.3565  3.0332  0.2529  0.1515    8.5730  38.0957   10.5868   
6   26.7897   13.7595  2.5521  0.4236  0.2174   29.6339  20.4560   -2.9292   
7   96.2327   46.5165  4.1540  0.0779  0.0390  110.3550  85.0486   43.1844   
8   46.7619   15.1993  2.5786  0.3377  0.1913   24.7548  43.8771   -6.6812   
9   62.7766   29.9104  3.3331  0.2475  0.1261  -33.9065  57.5848   23.7710   

    fAlpha    fDist  class  
0   6.3609  205.261      1  
1  76.9600  256.788      1  
2  10.4490  116.737      1  
3   4.6480  356.462      

In [3]:
X = data.drop('class', axis=1)
y = data['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
# Split data into features (X) and labels (y)
# Make train (70%) and test (30%)

In [4]:
train_d = pd.concat([X_train, y_train], axis=1)
gamma = train_d[train_d['class'] == 1]
hadron = train_d[train_d['class'] == 0]

gams = gamma.sample(n=len(hadron),random_state=42)
#take sample from gamma 3la ad 3dd el hadron

data_balanced = pd.concat([gams,hadron])
data_balanced = data_balanced.sample(frac=1,random_state=42) #mix data
print(data_balanced['class'].value_counts())
# Data has more Gamma than Hadron
# Take random sample from Gamma to match Hadron
# Mix them together


class
0    4686
1    4686
Name: count, dtype: int64


In [5]:
X_train = data_balanced.drop('class', axis=1)
y_train = data_balanced['class']
print("Balanced Data:\n", y_train.value_counts())

Balanced Data:
 class
0    4686
1    4686
Name: count, dtype: int64


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score


model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nDecision Tree")
print(confusion_matrix(y_test, y_pred))
#[[TN  FP]
#[FN  TP]]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))


Decision Tree
[[1588  414]
 [ 808 2896]]
Accuracy: 0.7858394672274799
Precision: 0.8749244712990937
Recall: 0.7818574514038877
F1: 0.8257770173937838


In [7]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import precision_score, recall_score, f1_score
nb = GaussianNB()
nb.fit(X_train, y_train)
#calc P(class = g) ,P(class = h),P(feature | class)
y_pred_nb = nb.predict(X_test)

print("\nNaive Bayes")
print(confusion_matrix(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))
print("F1:", f1_score(y_test, y_pred_nb))


Naive Bayes
[[ 851 1151]
 [ 398 3306]]
Accuracy: 0.7285313704872064
Precision: 0.741754543414853
Recall: 0.892548596112311
F1: 0.8101948290650656


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score

rf = RandomForestClassifier(random_state=42)

param_grid ={
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20]
}
#max depth decrease overfitting , n_estimaters increase power

grid_rf = GridSearchCV(rf, param_grid, cv=5)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred_rf = best_rf.predict(X_test)

print("\nRandom Forest")
print("Best params:", grid_rf.best_params_)
print("Best CV Score:", grid_rf.best_score_)
print(confusion_matrix(y_test, y_pred_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))


Random Forest
Best params: {'max_depth': 20, 'n_estimators': 200}
Best CV Score: 0.8583011881892565
[[1685  317]
 [ 441 3263]]
Accuracy: 0.8671573781983877
Precision: 0.9114525139664804
Recall: 0.880939524838013
F1: 0.8959362987369577


In [9]:
from sklearn.ensemble import AdaBoostClassifier

param_grid = {
    'n_estimators': [50, 100, 150]
}

grid_ada = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    param_grid,
    cv=5
)

grid_ada.fit(X_train, y_train)

best_ada = grid_ada.best_estimator_

y_pred_ada = best_ada.predict(X_test)

print("\nAdaBoost")
print("Best parameter:", grid_ada.best_params_)
print("Best CV Score:", grid_ada.best_score_)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("Accuracy:", accuracy_score(y_test, y_pred_ada))
print("Precision:", precision_score(y_test, y_pred_ada))
print("Recall:", recall_score(y_test, y_pred_ada))
print("F1:", f1_score(y_test, y_pred_ada))


AdaBoost
Best parameter: {'n_estimators': 150}
Best CV Score: 0.8109249662041977
Confusion Matrix:
 [[1625  377]
 [ 729 2975]]
Accuracy: 0.8061689449702067
Precision: 0.8875298329355609
Recall: 0.8031857451403888
F1: 0.8432539682539683


In [10]:
results = {
    "Model": ["Decision Tree", "Naive Bayes", "Random Forest", "AdaBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_ada),
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_ada),
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_ada),
    ],
    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_ada),
    ]
}
df_results = pd.DataFrame(results)
print("\n=== Final Comparison ===")
print(df_results)



=== Final Comparison ===
           Model  Accuracy  Precision    Recall        F1
0  Decision Tree  0.785839   0.874924  0.781857  0.825777
1    Naive Bayes  0.728531   0.741755  0.892549  0.810195
2  Random Forest  0.867157   0.911453  0.880940  0.895936
3       AdaBoost  0.806169   0.887530  0.803186  0.843254
